In [ ]:
import numpy as np
import wfdb
import pywt

import cv2
from scipy import signal

import os
import gc

In [ ]:
def windowed_data(data, wind_len=2500, num_leads=8):
    all_data = []
    for i in range(round(len(data) / wind_len)):
        cut_data = data[i*wind_len:i*wind_len + wind_len]
        if len(cut_data) < wind_len:
            cut_data = data[-wind_len:]
        all_data.append(cut_data)
    return np.array(all_data)[:,:,:8]

In [ ]:
def get_dwt_scalogram(window, wavelet='db4', level=5, shape=(125, 120)):
    h, w = shape
    ecg_data = window.T
    all_scalograms = []
    for lead_signal in ecg_data:

        coeffs = pywt.wavedec(lead_signal, wavelet, level=level)
        stack = [cv2.resize(np.abs(c).reshape(1, -1), (w, 1))[0] for c in coeffs[::-1]]
        scalogram = np.vstack(stack)
        if scalogram.shape[0] != h:
            scalogram = cv2.resize(scalogram, (w, h), interpolation=cv2.INTER_LINEAR)

        all_scalograms.append(np.log1p(scalogram))
    multi_channel_input = np.stack(all_scalograms, axis=-1)
    return multi_channel_input

def prepare_stft_data(window, fs=500, nperseg=256, noverlap=128):
    ecg_data = window.T
    all_spectrograms = []
    for lead_signal in ecg_data:
        
        f, t, Zxx = signal.stft(lead_signal, fs=fs, nperseg=nperseg, noverlap=noverlap)
        amplitude = np.abs(Zxx)
        
        spec_log = np.log10(amplitude + 1e-6)
        spec_norm = (spec_log - np.mean(spec_log)) / (np.std(spec_log) + 1e-8)
        all_spectrograms.append(spec_norm)
    multi_channel_input = np.stack(all_spectrograms, axis=-1)
    
    return multi_channel_input

Данные из Physionet Challenge 2020

In [ ]:
path_full = "cpsc_2018_extra\\"
g_list = ["g1\\","g2\\","g3\\","g4\\"]
RECORDS = []
for g in g_list:
    with open (path_full+g+"RECORDS", 'r', encoding='utf-8') as f:
        RECORDS += f.read().split("\n")[:-1]

In [ ]:
# all_codes = set()
# for g in g_list:
#     for rec_id in RECORDS:
#         try:
#             rec = wfdb.rdrecord(path_full+g+rec_id)
#         except FileNotFoundError:
#             # print(rec_ID)
#             continue
#         data_rec = rec.p_signal
#         header_rec = rec.comments
#         diag = header_rec[2][4:].split(",")
#         for i in diag:
#             all_codes.add(i)

# all_codes = list(all_codes)
# with open('all_codes_cpsc_2018_extra.txt', 'w', encoding='utf-8') as f:
#     f.write('\n'.join(all_codes))

with open('all_codes_cpsc_2018_extra.txt', 'r', encoding='utf-8') as f:
    all_codes = f.read().splitlines()

In [ ]:
def onehot_target(target, all_codes=all_codes):
    oh = np.zeros(len(all_codes))
    for i, code in enumerate(all_codes):
        if code in target:
            oh[i] = 1
    return oh

In [4]:
batch_size_records = 150 # Сохраняем каждые 150 пациентов
window_size = 2500
fs = 500
X_dwt_full, X_stft_full, y = [], [], []
data_dir = '/processed_data'

os.makedirs(data_dir, exist_ok=True)

In [ ]:
batch_count = 0

for g in g_list:
    for i, rec_id in enumerate(RECORDS):
        try:
            rec = wfdb.rdrecord(path_full+g+rec_id)
        except FileNotFoundError:
            continue
        
        data_rec = rec.p_signal
        
        header_rec = rec.comments
        diag = header_rec[2][4:].split(",")

        # if diag in rare_diags: slide = 100 else slide = window_size
        
        cut_data = windowed_data(data_rec, window_size)
        for window in cut_data:

            X_dwt_full.append(get_dwt_scalogram(window))
            X_stft_full.append(prepare_stft_data(window))
            y.append(onehot_target(diag))
        
        # Если накопили достаточно или это последний файл
        if (i + 1) % batch_size_records == 0:
            # Превращаем в массивы и сохраняем
            np.save(f'D:/processed_data/X_dwt_{batch_count}.npy', np.array(X_dwt_full, dtype=np.float32))
            np.save(f'D:/processed_data/X_stft_{batch_count}.npy', np.array(X_stft_full, dtype=np.float32))
            np.save(f'D:/processed_data/y_{batch_count}.npy', np.array(y, dtype=np.float32))
            
            # ОЧИСТКА
            X_dwt_full, X_stft_full, y = [], [], []
            gc.collect() # Принудительно очищаем память
            batch_count += 1
            print(f"Batch {batch_count} saved.")